# MvDEC 2025 canonical public gate: three datasets, seeds 42-44

Runs the versioned paper-informed full-batch reconstruction. The gate is fail-closed and staged: 20NEWS seed 42 first, then 20NEWS seeds 43-44, REUTERS, and RCV1 last. No failed or OOM run is replaced by a smaller batch or packaged as success.


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)
assert gpus, "Enable a Kaggle/Colab GPU before running this notebook."


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

TRAINING_COMMIT = "9b0c89afe522c009354bdd783b70fd182ec95d22"
REPO = Path("/kaggle/working/mvdec_fpmax")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "https://github.com/vdat-18/mvdec_fpmax.git", str(REPO)], check=True)
subprocess.run(["git", "checkout", "--detach", TRAINING_COMMIT], cwd=REPO, check=True)
resolved = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert resolved == TRAINING_COMMIT, (resolved, TRAINING_COMMIT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "gdown"], check=True)
sys.path.insert(0, str(REPO / "src"))
subprocess.run([
    sys.executable, str(REPO / "scripts/fetch_dekm_release.py"),
    "--datasets", "REUTERS", "20NEWS", "RCV1",
], cwd=REPO, check=True)

from sklearn.datasets import fetch_20newsgroups, fetch_rcv1
fetch_20newsgroups(subset="all", download_if_missing=True)
fetch_rcv1(download_if_missing=True)
print("Canonical public sources and exact training commit are ready.")


In [ ]:
import json
import os
from pipeline.public_reproduction_matrix import run_public_matrix

PROTOCOL = "mvdec_2025_paper_informed_full_batch_v2"
OUTPUT_ROOT = Path("/kaggle/working/mvdec-three-datasets-output")
ARCHIVE_ROOT = Path("/kaggle/working/mvdec-three-datasets-archives")
DATASET_ROOT = REPO / "external_repos/DEKM/datasets"


def run_stage(dataset, seeds):
    for seed in seeds:
        os.environ["PYTHONHASHSEED"] = str(seed)
        list(run_public_matrix(
            datasets=[dataset],
            protocols=[PROTOCOL],
            seeds=[seed],
            dataset_root=DATASET_ROOT,
            output_root=OUTPUT_ROOT,
            archive_root=ARCHIVE_ROOT,
            progress_interval=100,
            heartbeat_seconds=300,
            resume=True,
        ))


# Gate 1: the canonical sklearn 20NEWS source, seed 42 only.
run_stage("20NEWS", (42,))
seed42_dirs = sorted((OUTPUT_ROOT / "20news" / PROTOCOL).glob("*/seed_42"))
assert len(seed42_dirs) == 1, seed42_dirs
seed42_manifest = json.loads((seed42_dirs[0] / "manifest.json").read_text())
assert seed42_manifest["status"] == "complete"
assert seed42_manifest["training_commit"] == TRAINING_COMMIT
assert seed42_manifest["source_id"] == "sklearn_fetch_20newsgroups_all_tfidf2000_18846"
assert seed42_manifest["n_samples"] == 18_846

# Gate 2: continue 20NEWS only after seed 42 passed every audit.
run_stage("20NEWS", (43, 44))
run_stage("REUTERS", (42, 43, 44))

# RCV1 is intentionally last because it is the most expensive public run.
run_stage("RCV1", (42, 43, 44))


In [ ]:
import pandas as pd

runs_path = ARCHIVE_ROOT / "public_mvdec_matrix_runs.csv"
aggregate_path = ARCHIVE_ROOT / "public_mvdec_matrix_aggregate.csv"
runs = pd.read_csv(runs_path)
expected = {
    (dataset, seed)
    for dataset in ("20NEWS", "REUTERS", "RCV1")
    for seed in (42, 43, 44)
}
actual = set(zip(runs["dataset"], runs["seed"], strict=True))
assert actual == expected, (actual, expected)
assert runs["protocol_id"].eq(PROTOCOL).all()
assert runs["training_commit"].eq(TRAINING_COMMIT).all()
assert runs.loc[runs["dataset"] == "20NEWS", "source_id"].eq(
    "sklearn_fetch_20newsgroups_all_tfidf2000_18846"
).all()
aggregate = pd.read_csv(aggregate_path)
assert aggregate["completed_runs"].eq(3).all()
(ARCHIVE_ROOT / "failures.json").write_text("[]\n", encoding="utf-8")
display(runs[["dataset", "protocol_id", "seed", "source_id", "acc", "nmi", "runtime_seconds"]])
display(aggregate)


In [ ]:
import hashlib
import shutil

assert (ARCHIVE_ROOT / "failures.json").read_text(encoding="utf-8") == "[]\n"
zip_path = shutil.make_archive(
    "/kaggle/working/mvdec-v2-three-datasets-seeds42-44",
    "zip",
    ARCHIVE_ROOT,
)
digest = hashlib.sha256(Path(zip_path).read_bytes()).hexdigest()
print("ZIP:", zip_path)
print("SHA-256:", digest)
